In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display, Markdown
# package loading

In [2]:
# 데이터 전처리

# 데이터 로드
file_path = r'C:\Users\esang\Documents\work\devel\kdy\data\raw\pew2025.csv'
df = pd.read_csv(file_path)
print(df.head())

   caseid  folnews  seeknews  winpol  news_pay  paywall1  fundnprpbs  \
0       7        1         1       3         1         1           2   
1       9        1         1       3         2        -1           1   
2      11        1         1       1         2         4           2   
3      15        2         2       3         2         3           3   
4      21        2         3       3         2         5           1   

   soctrust4  chgtrst  trust_nhbr  ...  trust_msn  use_joe  trust_joe  \
0          2        4           2  ...          0        0          0   
1          2        4           2  ...          0        0          0   
2          2        3           2  ...          1        0          0   
3          3        4           3  ...          1       -1         -1   
4          3        4           3  ...          0        1          1   

   use_max  trust_max  newsattention  negemotion  posemotion  n_used  \
0        0          0            3.0        2.75        

### pew2025 '퓨리서치센터 미국 트렌드 패널 165차 데이터' 를 활용
> 미국인들의 뉴스 소비 방식과 신뢰도, 감정 반응에 대한 데이터이다.
> 이에 따라, 주제로 정치 당파성(partisanship)이 특정 언론사 신뢰도와 뉴스 소비 시의 감정 반응에 미치는 영향을 선정.

1. H1: 독립표본 t-검정 (Independent-samples t test)
    - 가설: 지지 정당(공화당 vs 민주당)에 따라 뉴스를 접할 때 느끼는 부정적 감정(negemotion) 수준에 통계적으로 유의미한 차이가 있을 것이다.
    - 이유: 정치 성향(partysum_final: 1=공화당, 2=민주당)이라는 명확히 구분되는 2개 집단 간에, 뉴스를 보며 느끼는 피로감이나 분노 등 연속형 변수인 부정적 감정 점수의 평균 차이를 비교하기에 독립표본 t-검정이 완벽하게 부합한다.
2. H2: 카이제곱 독립성 검정 (Chi-square test of independence)

    가설: 지지 정당(공화당 vs 민주당)에 따라 보수 성향 매체인 폭스뉴스(trust_fox)를 신뢰하는 비율에 유의미한 차이가 있을 것이다.

    이유: 지지 정당(범주형 2집단)과 폭스뉴스 신뢰 여부(범주형 2집단: 0=불신, 1=신뢰) 간의 연관성을 분할표를 통해 확인하는 분석입니다. 정치 양극화가 매체 신뢰도에 미치는 영향을 가장 직관적으로 보여줄 수 있습니다.

In [3]:
# 결측치 처리
df.replace([-1, 998, 999], np.nan, inplace=True)

,caseid,folnews,seeknews,winpol,news_pay,paywall1,fundnprpbs,soctrust4,chgtrst,trust_nhbr,...,trust_msn,use_joe,trust_joe,use_max,trust_max,newsattention,negemotion,posemotion,n_used,n_trusted
0,7.0,1.0,1.0,3.0,1.0,1.0,2.0,2.0,4.0,2.0,...,0.0,0.0,0.0,0.0,0.0,3.00,2.75,3.25,4,4
1,9.0,1.0,1.0,3.0,2.0,NaN,1.0,2.0,4.0,2.0,...,0.0,0.0,0.0,0.0,0.0,3.50,2.75,2.25,1,1
2,11.0,1.0,1.0,1.0,2.0,4.0,2.0,2.0,3.0,2.0,...,1.0,0.0,0.0,NaN,NaN,3.00,3.00,3.25,3,3
3,15.0,2.0,2.0,3.0,2.0,3.0,3.0,3.0,4.0,3.0,...,1.0,NaN,NaN,NaN,NaN,3.00,2.50,2.75,2,2
4,21.0,2.0,3.0,3.0,2.0,5.0,1.0,3.0,4.0,3.0,...,0.0,1.0,1.0,1.0,1.0,1.50,2.50,3.25,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1863,9451.0,1.0,1.0,3.0,1.0,3.0,2.0,2.0,4.0,2.0,...,0.0,NaN,NaN,NaN,NaN,3.75,2.75,3.00,2,2
1864,9453.0,1.0,3.0,3.0,1.0,2.0,2.0,3.0,4.0,3.0,...,0.0,0.0,0.0,NaN,NaN,3.25,4.00,3.25,0,1
1865,9473.0,2.0,3.0,3.0,2.0,3.0,2.0,2.0,3.0,2.0,...,0.0,0.0,0.0,NaN,NaN,2.50,3.00,2.75,1,2
1866,9476.0,2.0,3.0,3.0,1.0,3.0,3.0,4.0,5.0,1.0,...,NaN,NaN,NaN,NaN,NaN,3.00,1.75,3.00,0,0


## H1. 독립표본 t-검정 (지지 정당 -> 뉴스 시청 시 부정적 감정)

In [5]:
display(Markdown("### [H1] 지지 정당에 따른 뉴스 소비 시 부정적 감정 수준 차이 (t-test)"))

# 분석에 필요한 변수만 결측치 제거
h1_df = df.dropna(subset=['partysum_final', 'negemotion']).copy()
# partysum_final: 1=공화당(Republican), 2=민주당(Democrat)
group_rep = h1_df[h1_df['partysum_final'] == 1]['negemotion']
group_dem = h1_df[h1_df['partysum_final'] == 2]['negemotion']

# 1. Levene 등분산 검정
levene_stat, levene_p = stats.levene(group_rep, group_dem)
equal_var = levene_p >= 0.05

# 2. t-검정 수행
t_stat, p_val = stats.ttest_ind(group_rep, group_dem, equal_var=equal_var)

# 3. Cohen's d 계산
n1, n2 = len(group_rep), len(group_dem)
var1, var2 = group_rep.var(), group_dem.var()
pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
# 효과 크기 비교를 위해 절댓값으로 처리
cohens_d = abs(group_rep.mean() - group_dem.mean()) / pooled_std

t_test_results = pd.DataFrame({
    '집단 (지지 정당)': ['공화당', '민주당'],
    '표본 수(N)': [n1, n2],
    '부정적 감정 평균': [round(group_rep.mean(), 2), round(group_dem.mean(), 2)],
    '표준편차': [round(group_rep.std(), 2), round(group_dem.std(), 2)],
    't-통계량': [round(t_stat, 4), '-'],
    'p-value': [f"{p_val:.4f}", '-'],
    "Cohen's d": [round(cohens_d, 4), '-']
})

display(t_test_results)

### [H1] 지지 정당에 따른 뉴스 소비 시 부정적 감정 수준 차이 (t-test)

,집단 (지지 정당),표본 수(N),부정적 감정 평균,표준편차,t-통계량,p-value,Cohen's d
0,공화당,829,2.91,0.65,-15.0853,0.0000,0.6988
1,민주당,990,3.42,0.79,-,-,-


### H1: 지지 정당에 따른 뉴스 소비 시 부정적 감정 수준 차이 (독립표본 t-검정)
1. 결과 요약
    * 평균 차이: 민주당 지지자가 뉴스를 볼 때 느끼는 부정적 감정 점수(평균 3.42점)가 공화당 지지자(평균 2.91점)보다 높게 나타났다.
    * 통계적 유의성 (p=0.0000): p값이 기준치(0.05)보다 현저히 작으므로, 두 집단 간의 점수 차이는 통계적으로 매우 유의미하다.
    * 효과 크기 (Cohen's d = 0.6988): 약 0.7에 달하는 수치로, 지지 정당에 따른 감정 차이가 현실에서도 '중간에서 큰 수준(Medium-to-large)'으로 뚜렷하게 존재함을 의미한다.

2. 해석
    * 민주당 지지자들이 공화당 지지자들보다 뉴스를 소비할 때 분노, 피로감 등의 부정적인 감정을 유의미하게 더 많이 느끼는 것으로 확인됐다.
    * 이는 뉴스를 접하는 과정에서 경험하는 감정적 스트레스가 정치적 성향에 따라 상당히 큰 편차를 보이고 있음을 시사한다.
---

In [6]:
display(Markdown("### [H2] 지지 정당과 폭스뉴스(Fox News) 신뢰 여부의 연관성 (Chi-square)"))

h2_df = df.dropna(subset=['partysum_final', 'trust_fox']).copy()
h2_df['party'] = h2_df['partysum_final'].map({1: '공화당', 2: '민주당'})
h2_df['trust_fox_label'] = h2_df['trust_fox'].map({0.0: '신뢰 안 함', 1.0: '신뢰함'})

# 교차표 생성
contingency_table = pd.crosstab(h2_df['party'], h2_df['trust_fox_label'])

# 카이제곱 검정
chi2, p_val_chi, dof, expected = stats.chi2_contingency(contingency_table)

# Cramer's V (효과 크기) 계산
n_total = contingency_table.sum().sum()
k = min(contingency_table.shape)
cramers_v = np.sqrt(chi2 / (n_total * (k - 1)))

# 결과 표 만들기
chi2_results = pd.DataFrame({
    '분석 내용': ['지지 정당 vs Fox News 신뢰'],
    '총 표본 수(N)': [n_total],
    '최소 기대빈도': [round(expected.min(), 2)],
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도(df)': [dof],
    'p-value': [f"{p_val_chi:.4e}"], # 값이 너무 작으므로 지수 표기법 유지
    "Cramer's V": [round(cramers_v, 4)]
})

display(chi2_results)

display(Markdown("**▼ 지지 정당별 폭스뉴스 신뢰도 빈도 (교차표)**"))
display(contingency_table)

### [H2] 지지 정당과 폭스뉴스(Fox News) 신뢰 여부의 연관성 (Chi-square)

,분석 내용,총 표본 수(N),최소 기대빈도,카이제곱 통계량,자유도(df),p-value,Cramer's V
0,지지 정당 vs Fox News 신뢰,1730,295.21,453.9816,1,9.8088e-101,0.5123


**▼ 지지 정당별 폭스뉴스 신뢰도 빈도 (교차표)**

trust_fox_label,신뢰 안 함,신뢰함
party,,
공화당,289,509
민주당,801,131


### H2: 지지 정당과 폭스뉴스(Fox News) 신뢰 여부의 연관성 (카이제곱 독립성 검정)

1. 결과 요약
    * 통계적 유의성 (p=0.0000): p값이 0.05보다 압도적으로 작아, 지지 정당과 폭스뉴스 신뢰도 간에는 매우 강력한 통계적 연관성이 존재한다.
    * 효과 크기 (Cramer's V = 0.5123): 보통 0.5를 넘어가면 두 변수 간의 상관관계가 '매우 강함(Very strong)'을 나타낸다.
    * 빈도 분포(비율): 공화당 지지자는 폭스뉴스를 신뢰하는 비율이 압도적으로 높은 반면, 민주당 지지자는 거의 대부분이 폭스뉴스를 신뢰하지 않는 극단적인 쏠림 현상이 관찰되었다.

2. 해석
    * 개인의 지지 정당이 특정 보수 언론사(폭스뉴스)를 신뢰하는지 여부와 매우 강력하게 직결되어 있음이 확인되었다.
    * 이는 미국 사회의 미디어 소비 환경이 정치적 양극화에 따라 완전히 분절되어 있으며, 진영에 따라 '믿고 보는 언론'이 극단적으로 나뉘는 확증 편향적 생태계가 형성되어 있음을 뚜렷하게 보여준다.

## 결과 종합
1. 분석 결과의 종합적 의미
* 정치적 양극화(Political Polarization)가 미국인들의 매체 신뢰와 뉴스 소비 과정에서의 감정적 반응에 얼마나 깊이 개입하고 있는지를 명확히 보여준다.
* H1 결과에서 알 수 있듯, 민주당 지지자들(평균 3.42점)은 공화당 지지자들(평균 2.91점)보다 뉴스를 볼 때 분노나 피로감 같은 부정적인 감정을 훨씬 강하게 경험하고 있으며, 그 차이 역시 Cohen's d ≒ 0.70으로 상당히 큰 효과 크기를 나타냈다.
* 나아가 H2의 카이제곱 검정 결과(Cramer's V ≒ 0.51, 매우 강한 연관성), 공화당 지지자는 폭스뉴스를 높은 비율로 신뢰하는 반면 민주당 지지자는 절대다수가 불신하는 확연한 차이를 보였다. 즉, 지지 정당에 따라 '믿고 보는 매체'가 완전히 나뉘어져 있으며, 이러한 확증편향적 미디어 환경과 정치적 상황의 불일치가 특정 진영의 뉴스 스트레스와 부정적 감정을 가중시키고 있음을 종합적으로 추론해 볼 수 있다.

2. 분석의 한계점
* 첫째, 이 분석은 정당 지지율이나 특정 언론사에 대한 신뢰 여부 등 '단일 문항'을 기준으로 집단을 나누었기 때문에, 개인이 여러 매체를 교차하여 소비하는 복합적인 뉴스 이용 행태를 정교하게 반영하지 못했다.
* 둘째, 민주당 지지자들이 뉴스를 보며 부정적 감정을 더 느끼는 원인이 '본인이 신뢰하는 언론에서 부정적인 사회 이슈를 더 많이 다루기 때문'인지, 아니면 '자신의 진영과 반대되는 정치적 상황에 대한 피로감' 때문인지 구체적인 인과관계를 본 설문 데이터만으로는 확정 지을 수 없다.